In [ ]:
%pip install pandas kaggle trl accelerate unsloth

import os
import json
from kaggle.api.kaggle_api_extended import KaggleApi

if not os.path.exists("deep_past_data_past_data/train.csv"):
  if os.getenv("COLAB_RELEASE_TAG"):
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

  api = KaggleApi()
  api.config_values["username"] = os.environ["KAGGLE_USERNAME"]
  api.config_values["key"] = os.environ["KAGGLE_API_TOKEN"]

  api.competition_download_files("deep-past-initiative-machine-translation")
  !unzip deep-past-initiative-machine-translation.zip -d deep_past_data_past_data

import pandas as pd

train_df = pd.read_csv("deep_past_data_past_data/train.csv")
test_df = pd.read_csv("deep_past_data_past_data/test.csv")
definitions_df = pd.read_csv("deep_past_data_past_data/eBL_Dictionary.csv")

Archive:  deep-past-initiative-machine-translation.zip
  inflating: deep_past_data_past_data/OA_Lexicon_eBL.csv  
  inflating: deep_past_data_past_data/Sentences_Oare_FirstWord_LinNum.csv  
  inflating: deep_past_data_past_data/bibliography.csv  
  inflating: deep_past_data_past_data/eBL_Dictionary.csv  
  inflating: deep_past_data_past_data/publications.csv  
  inflating: deep_past_data_past_data/published_texts.csv  
  inflating: deep_past_data_past_data/resources.csv  
  inflating: deep_past_data_past_data/sample_submission.csv  
  inflating: deep_past_data_past_data/test.csv  
  inflating: deep_past_data_past_data/train.csv  


In [34]:
train_df

,oare_id,transliteration,translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū..."
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...
...,...,...,...
1556,ff3208e4-8ab8-4368-b4df-7b80afa5bc32,um-ma en-nam-a-šur-ma a-na a-la-ḫi-im qí-bi-ma...,From Ennam-Aššur to Ali-ahum: Here 2 men have ...
1557,ff43a284-3d67-4238-8b4a-9b6cb7531e0a,3 ma-na KÙ.BABBAR ṣa-ru-pá-am i-na ší-im SÍG.Ḫ...,Ilī-ašrannī son of Sukkalliya has received 3 m...
1558,ff5747a4-af8a-4100-a906-a2660ae72606,ša-lim-a-šùr a-na a-mur-IŠTAR ú-ṭá-ḫi-ni-a-tí-...,Šalim-Aššur made us approach Amur-Ištar and Ša...
1559,ff777871-97ce-4bfc-bdfb-73352868944d,a-na en-nam-a-šùr qí-bi-ma um-ma IŠTAR-ra-bi₄-...,To Ennam-Aššur from Ištar-rabiʾat: With respec...


In [ ]:
definitions_df

In [ ]:
import re
in_quotes = r'"([^"]*)"' # regex to capture text within quotes

definitions_df["translation"] = definitions_df["definition"].fillna("").apply(
    lambda x: re.findall(in_quotes, x)[0] if re.findall(in_quotes, x) else ""
)
definitions_df["transliteration"] = definitions_df["word"]
definitions_df = definitions_df[["transliteration", "translation"]]
definitions_df

In [ ]:
def create_text_column(row):
    if "translation" not in row or pd.isna(row["translation"]):
        return "Akkadian:" + row["transliteration"] + ":" + "English:"
    return "Akkadian:" + row["transliteration"] + ":" + "English: " + row["translation"]

In [ ]:
train_df["text"] = train_df.apply(create_text_column, axis=1)
test_df["text"] = test_df.apply(create_text_column, axis=1)
definitions_df["text"] = definitions_df.apply(create_text_column, axis=1)

validation_df = train_df.sample(frac=0.1, random_state=0)
train_df = train_df.drop(validation_df.index)

from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
validation_dataset = Dataset.from_pandas(validation_df)
test_dataset = Dataset.from_pandas(test_df)
definitions_dataset = Dataset.from_pandas(definitions_df)

In [ ]:
from unsloth import FastLanguageModel
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig
import torch

# Safe in spawned processes; will crash if the launcher uses fork.
bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3-4B",
    dtype = torch.bfloat16 if bf16_ok else torch.float16,
    trust_remote_code = True,
    max_seq_length = 1024,
    load_in_4bit = True,
    full_finetuning = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32
    , # This controls the rank of the LoRA update. Increase for larger models/datasets for better quality.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 512, # This controls the scaling of the LoRA update. Usually set to r*16.
    lora_dropout = 0.0, # Dropout for LoRA layers.
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 8888,
    use_rslora = True,
    loftq_config = None,
)

# First train on definitions to give model basic vocabulary
definitions_training_args = SFTConfig(
    output_dir="akkadian-definitions-model-dictionary",
    dataset_text_field="text",
    report_to = "none", # Use TrackIO/WandB etc
    num_train_epochs=5,
    per_device_train_batch_size=2048,
    # load_best_model_at_end=True,
    # batch_eval_metrics=True,
    # eval_strategy="epoch",
    # save_strategy="epoch",
)

definitions_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=definitions_dataset,
    # eval_dataset=validation_dataset,
    args=definitions_training_args,
)
definitions_trainer.train()
definitions_trainer.save_model(definitions_training_args.output_dir)

In [ ]:
# Safe in spawned processes; will crash if the launcher uses fork.
bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = definitions_training_args.output_dir,
    dtype = torch.bfloat16 if bf16_ok else torch.float16,
    trust_remote_code = True,
    max_seq_length = 1024,
    load_in_4bit = True,
    full_finetuning = False
)

training_args = SFTConfig(
    output_dir="akkadian-translation-model",
    dataset_text_field="text",
    report_to = "none", # Use TrackIO/WandB etc
    num_train_epochs=3,
    per_device_train_batch_size=1024,
    load_best_model_at_end=True,
    batch_eval_metrics=True,
    eval_strategy="epoch",
    save_strategy="epoch",
)

# Then train on main translation dataset
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    args=training_args,
)
trainer.train()
trainer.save_model(training_args.output_dir)

In [ ]:
# Load the model and tokenizer using Unsloth for inference
model_path = "akkadian-translation-model"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 512,
    dtype = None, # Auto-detect
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# Function to generate translation from the text column using the optimized model
def generate_translation(text):
    inputs = tokenizer([text], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 100, use_cache = True)
    # Decode only the generated part by skipping the prompt tokens
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return generated_text.strip()

# Generate predictions
# We use the 'text' field which contains the prompt format defined in create_text_column
debug_df = train_df.head(10).copy()

def remove_translation_part(text):
    # Remove the "English: " part to only keep the Akkadian transliteration
    if "English:" in text:
        return text.split("English:")[0] + "English:"
    return text

debug_df["text"] = debug_df["text"].apply(remove_translation_part)

debug_df["predicted_translation"] = debug_df["text"].apply(generate_translation)
debug_df["id"] = debug_df.index

# Save result
submission_df = debug_df[["id", "predicted_translation"]]
submission_df.to_csv("akkadian_translations.csv", index=False)

out = debug_df[["transliteration", "translation", "predicted_translation"]]
out.to_csv("akkadian_debug_output.csv", index=False)

out